helix by protein
AUC : 0.6414825647553142
PRC : 0.5849129035397901
protein_0,1527 : 0.5151624029893648
protein_1,3002 : 0.5023551699270842
protein_2,4401 : 0.4460576607115062
protein_3,5786 : 0.541838787488159
protein_4,7216 : 0.5591760435061244
protein_5,8587 : 0.5238090255981028
protein_6,9872 : 0.4941701072145433
protein_7,11130 : 0.5229911697884297
protein_8,12278 : 0.6409474401551072
protein_9,13397 : 0.4973395780501082
protein_10,14409 : 0.4815858081797427
protein_11,15433 : 0.5498405569069441
protein_12,16469 : 0.8252332580887886
protein_13,17502 : 0.4953459818016934
protein_14,18479 : 0.5994407383017915
protein_15,19408 : 0.6366348911105943
helix layer 1
AUC : 0.6094402549384897
PRC : 0.5525694087103843
concat
AUC : 0.6564208143347434
PRC : 0.5849318201908431
protein_0,1527 : 0.5413844675194022
protein_1,3002 : 0.600608395050423
protein_2,4401 : 0.529723197454212
protein_3,5786 : 0.5137183956874579
protein_4,7216 : 0.5260331599597403
protein_5,8587 : 0.5285636232591662
protein_6,9872 : 0.6291674146118573
protein_7,11130 : 0.8433796304244568
protein_8,12278 : 0.625259151901407
protein_9,13397 : 0.4727634531380698
protein_10,14409 : 0.542064048119498
protein_11,15433 : 0.5538441426256985
protein_12,16469 : 0.7399209932279909
protein_13,17502 : 0.5708394045302168
protein_14,18479 : 0.4694505683282274
protein_15,19408 : 0.6060179803146555
byprotein
AUC : 0.6356664116366864
PRC : 0.5608192555988003
protein_0,1527 : 0.46655603262431733
protein_1,3002 : 0.5979737335119026
protein_2,4401 : 0.49133965168364435
protein_3,5786 : 0.4946386195177687
protein_4,7216 : 0.46753212835672775
protein_5,8587 : 0.5269059077910302
protein_6,9872 : 0.6902364594722499
protein_7,11130 : 0.7917603767862151
protein_8,12278 : 0.6475846082967002
protein_9,13397 : 0.4937945616527478
protein_10,14409 : 0.5568726139239601
protein_11,15433 : 0.44827554007554293
protein_12,16469 : 0.6796726862302482
protein_13,17502 : 0.48269581520465454
protein_14,18479 : 0.547484753915309
protein_15,19408 : 0.5854413702239789
random
AUC : 0.7982324257952687
PRC : 0.748602375428713

In [3]:
from helix_model import *
import numpy as np
import random
import time
import timeit

def load_tensor(file_name, dtype):
    return [dtype(d).to(device) for d in np.load(file_name + '.npy',allow_pickle=True)]

def load_tensor2(file_name, dtype):
    proteins = np.load(file_name + '.npy', allow_pickle=True)
    torch_tensor_protein = []
    for protein in proteins:
        this_protein = []
        for helix in protein:
            this_protein.append(dtype(helix).to(device))
        torch_tensor_protein.append(this_protein)
    return torch_tensor_protein


def shuffle_dataset(dataset, seed):
    np.random.seed(seed)
    np.random.shuffle(dataset)
    return dataset


def split_dataset(dataset, ratio):
    n = int(ratio * len(dataset))
    dataset_1, dataset_2 = dataset[:n], dataset[n:]
    return dataset_1, dataset_2


from word2vec import seq_to_kmers, get_protein_embedding
from gensim.models import Word2Vec
import torch
import os

"""CPU or GPU"""
if torch.cuda.is_available():
    device = torch.device('cuda:0')
    print('The code uses GPU...')
else:
    device = torch.device('cpu')
    print('The code uses CPU!!!')

""" create model ,trainer and tester """
protein_dim = 100
atom_dim = 34
hid_dim = 64
n_layers = 3
n_heads = 8
pf_dim = 256
dropout = 0.1
batch = 64
lr = 1e-4
weight_decay = 1e-4
decay_interval = 5
lr_decay = 1.0
iteration = 90
kernel_size = 7

encoder = Encoder(protein_dim, hid_dim, n_layers, kernel_size, dropout, device)
decoder = Decoder(atom_dim, hid_dim, n_layers, n_heads, pf_dim, DecoderLayer, SelfAttention, PositionwiseFeedforward, dropout, device)
model = Predictor(encoder, decoder, device)

The code uses GPU...


In [4]:
path = "/home/yamane/helixEncoder/helix_encoder/output/model/"

byprotein = "lr=1e-4,dropout=0.1,weight_decay=1e-4,kernel=7,n_layer=3,batch=64,byprotein,90e"
ran = "lr=1e-4,dropout=0.1,weight_decay=1e-4,kernel=7,n_layer=3,batch=64"
#helix = "lr=1e-4,dropout=0.1,weight_decay=1e-4,kernel=7,n_layer=3,batch=64,ClassAbyproteinHelix,100e,concat"
helix = "20220628_dataset10_helixencoder"

indexs = [
1527 ,
3002 ,
4401 ,
5786 ,
7216 ,
8587 ,
9872 ,
11130 ,
12278 ,
13397 ,
14409 ,
15433 ,
16469 ,
17502 ,
18479 ,
19408
]

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import math
import numpy as np
from sklearn.metrics import roc_auc_score, precision_score, recall_score, precision_recall_curve, auc
from Radam import *
from lookahead import Lookahead

#DATASET = "classAByProteinWithHelix_test"
#DATASET = "byProteinWithHelix_concat_test.txt"
DATASET = "classA_bp_test"
model.to(device)
#lambda storage,loc:storage
model.load_state_dict(torch.load(path + helix,map_location = device))

dir_input = ('dataset/' + DATASET + '/word2vec_30/')
compounds = load_tensor(dir_input + 'compounds', torch.FloatTensor)
adjacencies = load_tensor(dir_input + 'adjacencies', torch.FloatTensor)
proteins = load_tensor2(dir_input + 'proteins', torch.FloatTensor)
interactions = load_tensor(dir_input + 'interactions', torch.LongTensor)

dataset = list(zip(compounds, adjacencies, proteins, interactions))
print(len(dataset))

19409


In [9]:

tester = Tester(model)

In [10]:
AUC, PRC = tester.test(dataset)
print(f"AUC : {AUC}")
print(f"PRC : {PRC}")

AUC : 0.6144196409681075
PRC : 0.5613856659223078


In [ ]:
# import pickle
# output = "normal.dump"
# with open("sigbio_result/" + output,"wb") as f:
#   pickle.dump([T,S],f)

# import matplotlib.pyplot as plt
# from sklearn.metrics import roc_curve

# with open("sigbio_result/helix.dump","rb") as f:
#   helix = pickle.load(f)

# with open("sigbio_result/concat.dump","rb") as f:
#   concat = pickle.load(f)

# with open("sigbio_result/normal.dump","rb") as f:
#   normal = pickle.load(f)

# fpr_h, tpr_h, thresholds_h = roc_curve(helix[0], helix[1])
# fpr_c, tpr_c, thresholds_c = roc_curve(concat[0], concat[1])
# fpr_n, tpr_n, thresholds_n = roc_curve(normal[0], normal[1])

# plt.plot(fpr_h, tpr_h, color='red',label = "TransformerCPI replaced Helix encoder")
# plt.plot(fpr_c, tpr_c, color='blue',label = "TrasformerCPI (Helix sequences)")
# plt.plot(fpr_n, tpr_n, color='green',label = "TrasformerCPI (All protein sequences)")

# plt.plot([0, 1], [0, 1],linestyle='--')
# plt.xlabel('False Positive Rate')
# plt.ylabel('True Positive Rate')
# plt.title('Receiver Operating Characteristic Curve')
# plt.legend()
# plt.savefig('roc_curve.png')
# plt.show()

In [ ]:
# AUCs = []
# prenum = 0
# for p,i in enumerate(indexs):
#   try:
#     _dataset = dataset[prenum:i+1]
#     AUC = predict(model,_dataset)
#     AUCs.append(AUC)
#     prenum = i+1
#     print(f"protein_{p},{i} : {AUC}")
#   except:
#     print("e")
# import pickle

# with open("helix.dump","wb") as f:
#   y_true = np.array(T)
#   y_score= np.array(S)
#   pickle.dump([y_true,y_score],f)

# from sklearn.metrics import roc_curve
# import matplotlib.pyplot as plt
# import numpy as np

# #T : correct
# #Y : predict

# y_true = np.array(T)
# y_score= np.array(S)
# fpr, tpr, thresholds = roc_curve(y_true, y_score)
# plt.plot(fpr, tpr, color='red')
# plt.plot([0, 1], [0, 1], color='green', linestyle='--')
# plt.xlabel('False Positive Rate')
# plt.ylabel('True Positive Rate')
# plt.title('Receiver Operating Characteristic Curve')
# plt.legend()
# plt.savefig('normal_roc_curve.png')
# plt.show()

In [ ]:
# import matplotlib.pyplot as plt
# result = [0.641,0.656,0.636]
# labels = ["TransformerCPI \n replaced Helix encoder","TrasformerCPI \n (Helix sequences)","TrasformerCPI \n (All protein sequences)"]
# x = np.arange(len(labels))
# width = 0.50

# fig, ax = plt.subplots()
# rect = ax.bar(x, result, width)
# ax.set_xticks(x)
# ax.set_xticklabels(labels)

# def autolabel(rects):
#     for rect in rects:
#         height = rect.get_height()
#         ax.annotate('{}'.format(height),
#                    xy=(rect.get_x() + rect.get_width() / 2, height),
#                    xytext=(0, 3),
#                    textcoords="offset points",
#                    ha='center', va='bottom')
# autolabel(rect)
# #plt.xlabel('model')
# plt.title('AUC on test data for each model')
# plt.ylabel('AUC')
# plt.ylim(0.50, 0.75)
# plt.savefig('3model_result.png')   
# plt.show()

In [ ]:
# re_path = "protein_result/re_202202.txt"
# protein_result = {}
# with open(re_path,"r") as f:
#   for file in f:
#     key = file.split(":")[0]
#     value = float(file.split(":")[1].replace("\n",""))
#     if key in protein_result:
#       protein_result[key].append(value)
#     else:
#       protein_result[key] = [value]

# ids = ['P0DMS8',
#  'P08908',
#  'P50406',
#  'P28223',
#  'P07550',
#  'P61169',
#  'P29274',
#  'P21453',
#  'P30542',
#  'Q9Y5N1',
#  'P08588',
#  'P21917',
#  'Q99500',
#  'P33533',
#  'P19327',
#  'P13945']

# import pandas as pd

# file_path = "/data/yamane/GPCRclassAData/"
# fle_name = ["chemblSmiles.csv","classA_ligand_binary_202110.csv","SHP_for_featurize.csv"]
# DataPath = "/home/yamane/transformerCPI/data/classAByProtein/"

# test_df = pd.read_csv(DataPath+"byProtein_test.csv",index_col = 0)
# seq_df = pd.read_csv(file_path+fle_name[2]).drop("Unnamed: 0",axis = 1)

# ids = []
# id2length = {}
# seq_df_all = pd.read_csv(file_path+"id2seq.csv").drop("Unnamed: 0",axis = 1)
# for i in range(len(test_df)):
#   id = test_df.iloc[i]["UniProt ID"]
#   if id not in ids:
#     ids.append(id)
#     seq = len("".join([seq_df[seq_df["id"] == id]["h"+str(i+1)].values[0] for i in range(7)]))
#     all = seq_df_all[seq_df_all["UniProt ID"] == id]["sequence"].values[0]
#     #print(f"id : {id},seq : {seq} , all {len(all)} ")
#     #print(f"{id} div {len(all) - seq}")
#     id2length[id] = [seq,len(all),len(all) - seq]


# font_s = "{\\scriptsize "
# font_e = "}"
# print(f"uniprot id & {font_s}Helix encoder{font_e} & {font_s}input : Helix sequences{font_e} & {font_s}input : All protein sequences{font_e}\\\\\\hline\\hline")
# all_result = []
# for index,l in enumerate(protein_result):
#   values = protein_result[l]
#   all_result.append([ids[index],values[0],values[1],values[2],id2length[ids[index]][0],id2length[ids[index]][1],id2length[ids[index]][2]])
#   #print(f"id  : {ids[index]} >> helix : {values[0]} , helix_concat : {values[1]} , normal : {values[2]}")
#   #print(f"|{ids[index]}|{round(values[0], 3)} | {round(values[1], 3)} | {round(values[2], 3)}|")
#   hutozis_s = "\\bf{"
#   hutozi_e = "}"
#   hutozis = ["","","","","",""]
#   values = [round(values[0], 3),round(values[1], 3),round(values[2], 3)]
#   max_index = np.argmax(values)
#   if max_index == 0:
#     hutozis[0] = hutozis_s
#     hutozis[1] = hutozi_e
#   elif max_index == 1:
#     hutozis[2] = hutozis_s
#     hutozis[3] = hutozi_e
#   else:
#     hutozis[4] = hutozis_s
#     hutozis[5] = hutozi_e
#   print(f"{ids[index]} & {hutozis[0]}{round(values[0], 3)}{hutozis[1]} & {hutozis[2]}{round(values[1], 3)}{hutozis[3]} & {hutozis[4]}{round(values[2], 3)}{hutozis[5]}\\\\ \

In [ ]:
# re_path = "protein_result/re_202202.txt"
# protein_result = {}
# with open(re_path,"r") as f:
#   for file in f:
#     key = file.split(":")[0]
#     value = float(file.split(":")[1].replace("\n",""))
#     if key in protein_result:
#       protein_result[key].append(value)
#     else:
#       protein_result[key] = [value]

# ids = ['P0DMS8',
#  'P08908',
#  'P50406',
#  'P28223',
#  'P07550',
#  'P61169',
#  'P29274',
#  'P21453',
#  'P30542',
#  'Q9Y5N1',
#  'P08588',
#  'P21917',
#  'Q99500',
#  'P33533',
#  'P19327',
#  'P13945']

# import pandas as pd

# file_path = "/data/yamane/GPCRclassAData/"
# fle_name = ["chemblSmiles.csv","classA_ligand_binary_202110.csv","SHP_for_featurize.csv"]
# DataPath = "/home/yamane/transformerCPI/data/classAByProtein/"

# test_df = pd.read_csv(DataPath+"byProtein_test.csv",index_col = 0)
# seq_df = pd.read_csv(file_path+fle_name[2]).drop("Unnamed: 0",axis = 1)

# ids = []
# id2length = {}
# seq_df_all = pd.read_csv(file_path+"id2seq.csv").drop("Unnamed: 0",axis = 1)
# for i in range(len(test_df)):
#   id = test_df.iloc[i]["UniProt ID"]
#   if id not in ids:
#     ids.append(id)
#     seq = len("".join([seq_df[seq_df["id"] == id]["h"+str(i+1)].values[0] for i in range(7)]))
#     all = seq_df_all[seq_df_all["UniProt ID"] == id]["sequence"].values[0]
#     #print(f"id : {id},seq : {seq} , all {len(all)} ")
#     #print(f"{id} div {len(all) - seq}")
#     id2length[id] = [seq,len(all),len(all) - seq]


# #print("|uniprot id | helix encoder | concat input | normal |")
# #print("|---|---|---|---|")
# all_result = []
# for index,l in enumerate(protein_result):
#   values = protein_result[l]
#   all_result.append([ids[index],values[0],values[1],values[2],id2length[ids[index]][0],id2length[ids[index]][1],id2length[ids[index]][2]])
#   #print(f"id  : {ids[index]} >> helix : {values[0]} , helix_concat : {values[1]} , normal : {values[2]}")
#   #print(f"|{ids[index]}|{round(values[0], 3)} | {round(values[1], 3)} | {round(values[2], 3)}|")


# def _sort(A,B):
#   new = {}
#   #print(B)
#   for a,b in zip(A,B):
#     new[a] = b
#   new = sorted(new.items(), key=lambda x:x[1])
#   newa = [i[0] for i in new]
#   newb = [i[1] for i in new]
#   return newa,newb
  


# ids = [i[0] for i in all_result]
# helix = [i[1] for i in all_result]
# concat = [i[2] for i in all_result]
# normal= [i[3] for i in all_result]
# helix_length= [i[4] for i in all_result]
# all_length= [i[5] for i in all_result]
# dev= [i[6] for i in all_result]

# print(dev)
# X = dev
# helix,x = _sort(helix,X)
# concat,x = _sort(concat,X)
# normal,x = _sort(normal,X)

# print(dev)
# print(x)
# import matplotlib.pyplot as plt

# fig = plt.figure(figsize = (9,6))
# ax = fig.add_subplot(1, 1, 1)

# labels = ["Helix encoder","helix sequences","TrasformerCPI"]
# ax.plot(x, helix,  c="#ff0000",  marker='o',label=labels[0])
# ax.plot(x, concat, c="#0000ff",  marker='o',label=labels[1])
# ax.plot(x, normal, c="#00ff00",  marker='o',label=labels[2])
# ax.grid(axis='both')

# ax.set_xlabel("length of helix")
# ax.set_ylabel("protein AUC")
# ax.set_title("AUC-helix_length")
# ax.legend()
# #fig.savefig("AUC-helix_length.png")